[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anita-agasaveeran/Assignment4/blob/main/01_CRISP-DM_Bakery_Waste/notebooks/CRISP_DM_Bakery_Waste.ipynb)

# Reducing Bakery Over-Production with CRISP-DM
**CMPE 255 · Assignment 4 · Project 1 of 3**

This notebook follows all six phases of CRISP-DM 1.0 (Chapman et al., 2000) and every generic task in each phase. Each task ends with the outputs the CRISP-DM user guide lists for it.

| Phase | Tasks |
|---|---|
| 1. Business Understanding | Determine business objectives · Assess situation · Determine data mining goals · Produce project plan |
| 2. Data Understanding | Collect initial data · Describe data · Explore data · Verify data quality |
| 3. Data Preparation | Select · Clean · Construct · Integrate · Format data |
| 4. Modeling | Select modeling technique · Generate test design · Build model · Assess model |
| 5. Evaluation | Evaluate results · Review process · Determine next steps |
| 6. Deployment | Plan deployment · Plan monitoring & maintenance · Produce final report · Review project |

Supporting code lives in `src/bakery/`. The written report is `reports/final_report.md`, and critic reviews for each phase are in `critiques/`.

## 0. Setup

In [1]:
# Works in Google Colab and locally. In Colab: clone the repo and install requirements.
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/anita-agasaveeran/Assignment4.git"

if IN_COLAB:
    if not Path("/content/Assignment4").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Assignment4"], check=True)
    PROJECT_DIR = Path("/content/Assignment4/01_CRISP-DM_Bakery_Waste")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "/content/Assignment4/requirements.txt"], check=True)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_DIR / "src"))
os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)

Project dir: /Users/anitaagasaveeran/Desktop/MSSE_Fall2026/CMPE255/Assignment4/01_CRISP-DM_Bakery_Waste


In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from bakery import config
from bakery.data import load_raw
from bakery import metrics, policy

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(config.SEED)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | sklearn {sklearn.__version__}")

python 3.12.14 | pandas 3.1.0 | sklearn 1.9.1


---
# Phase 1 · Business Understanding

> **Revision 2**, after critic round 1 (score 5/10; see `critiques/phase1_business_understanding_round1.md`). The round-1 draft measured waste against the recorded `Units_Produced` field. The critic showed, and I confirmed, that this field is generated per order line as *ordered quantity + at least 1*, so it isn't a real production decision. Phase 1 has been reframed around a decision the data can support. This loop from data back to business understanding is CRISP-DM working as intended.

### 1.1 Determine business objectives

**CRISP-DM outputs:** Background · Business objectives · Business success criteria

#### Background
A three-store bakery chain sells 35 products. **22 are fresh** (median shelf life ≤ 7 days): cakes, pastries, bread, buns, milk, paneer, cold coffee and juice. Anything unsold is thrown away. The other 13 are **long-life stock** (biscuits, rusk, honey, jam, tea powder, butter, cheese), which keeps for 30+ days and is assumed to sell later (A6).

Management's concern: *"Our fresh range doesn't seem to make money once you count what we throw away. Should we keep baking all of it every day, and if so, how much?"* The cells below look at what the data can and can't say about this.

In [3]:
# Fresh scope and demand (Quantity = units customers ordered and paid for)
from bakery.data import fresh_products

df = load_raw()
FRESH = fresh_products(df)
df["is_fresh"] = df["Product"].isin(FRESH)
fresh = df[df["is_fresh"]].copy()

summary = (df.groupby(np.where(df["is_fresh"], "fresh (22 products)", "long-life (13 products)"))
             .agg(lines=("Order_ID", "size"), units_ordered=("Quantity", "sum"),
                  revenue=("Total_Bill", "sum"), reported_profit=("Profit", "sum")))
summary["share_of_revenue"] = summary["revenue"] / summary["revenue"].sum()
display(summary.round(2))

n_days = df["Date"].nunique()
print(f"{n_days} days. Orders/day across the whole chain: {df['Order_ID'].nunique() / n_days:.1f}; "
      f"fresh units/day across the chain: {fresh['Quantity'].sum() / n_days:.1f}")

,lines,units_ordered,revenue,reported_profit,share_of_revenue
fresh (22 products),12124,25432,1661466.53,674566.81,0.62
long-life (13 products),7197,17595,1029339.52,469170.74,0.38


731 days. Orders/day across the whole chain: 9.0; fresh units/day across the chain: 34.8


In [4]:
# Can the recorded production / unsold fields tell us how much is wasted? (risk R1)
checks = pd.Series({
    "Profit = (Selling_Price - Cost_Price) x Quantity": np.isclose(df["Profit"], (df["Selling_Price"] - df["Cost_Price"]) * df["Quantity"], atol=0.05).mean(),
    "Total_Bill = Selling_Price x Quantity": np.isclose(df["Total_Bill"], df["Selling_Price"] * df["Quantity"], atol=0.05).mean(),
    "Units_Produced - Quantity >= 1": ((df["Units_Produced"] - df["Quantity"]) >= 1).mean(),
    "Unsold_Units == 1": (df["Unsold_Units"] == 1).mean(),
    "Units_Sold > Quantity": (df["Units_Sold"] > df["Quantity"]).mean(),
})
display(checks.map("{:.1%}".format).to_frame("share of rows"))

# Fresh 'profit after waste' under the three possible readings of the recorded fields
m, c = fresh["Selling_Price"] - fresh["Cost_Price"], fresh["Cost_Price"]
readings = pd.DataFrame({
    "demand": ["Quantity", "Quantity", "Units_Sold"],
    "waste": ["Unsold_Units (as recorded)", "Units_Produced - Quantity", "Unsold_Units"],
    "fresh profit after waste (₹, 2 yrs)": [
        (m * fresh["Quantity"]).sum() - (c * fresh["Unsold_Units"]).sum(),
        (m * fresh["Quantity"]).sum() - (c * (fresh["Units_Produced"] - fresh["Quantity"])).sum(),
        (m * fresh["Units_Sold"]).sum() - (c * fresh["Unsold_Units"]).sum()]},
    index=["mixed (round-1 draft)", "consistent with Quantity", "consistent with Units_Sold"])
display(readings.round(0))

,share of rows
Profit = (Selling_Price - Cost_Price) x Quantity,100.0%
Total_Bill = Selling_Price x Quantity,100.0%
Units_Produced - Quantity >= 1,100.0%
Unsold_Units == 1,91.8%
Units_Sold > Quantity,26.7%


,demand,waste,"fresh profit after waste (₹, 2 yrs)"
mixed (round-1 draft),Quantity,Unsold_Units (as recorded),-24053.0
consistent with Quantity,Quantity,Units_Produced - Quantity,-304929.0
consistent with Units_Sold,Units_Sold,Unsold_Units,161826.0


**What the recorded fields can't tell us.** Production is logged **per order line** as the ordered quantity plus at least one unit (exactly one on about 92% of rows). There is no production on days without orders, and no stockout is ever recorded. That is a byproduct of how the data was generated, not a production decision anyone made, and depending on how you read it, fresh products either lose ₹305k or make ₹162k. **So this project doesn't use the recorded production/unsold fields as its baseline.** It uses only `Quantity` (what customers ordered and paid for) and simulates the bake decision explicitly.

A realistic current practice is assumed instead (A0): **every fresh product is baked ahead daily at every store**, in quantities a manager sets from recent sales. The next cell prices that practice, and the alternatives, on two out-of-sample festive quarters. Each rule and each set of unit economics is fit only on data before its quarter.

In [5]:
# Sizing the problem with ex-ante rules (no model): what would a store manager's rule of thumb earn?
from bakery.features import demand_grid, add_trailing_mean
from bakery.policy import simulate_plan, trailing_mean_rule, fit_multiplier

def unit_economics(frame):
    # Quantity-weighted unit margin and unit cost per product
    g = frame.assign(m=(frame.Selling_Price - frame.Cost_Price) * frame.Quantity,
                     c=frame.Cost_Price * frame.Quantity).groupby("Product")[["m", "c", "Quantity"]].sum()
    return pd.DataFrame({"unit_margin": g.m / g.Quantity, "unit_cost": g.c / g.Quantity})

# Two out-of-sample quarters, each with rules and economics fit only on data before the quarter
QUARTERS = {"Q4 2023": ("2023-10-01", "2023-12-31"), "Q4 2024": ("2024-10-01", "2024-12-31")}
RETENTION = [0.5, 0.7, 0.9]   # share of demand kept if made to order only (A7; applied to all fresh items here for sizing)
rows = []
for pooled in (False, True):
    grid = add_trailing_mean(demand_grid(fresh, item="Product", pooled=pooled))
    for qname, (q0, q1) in QUARTERS.items():
        fit_end = pd.Timestamp(q0) - pd.Timedelta(days=1) if qname == "Q4 2023" else pd.Timestamp(config.TRAIN_END)
        econ = unit_economics(fresh[fresh["Date"] <= fit_end])
        g = grid.join(econ, on="Product")
        tr = g[(g["Date"] <= fit_end) & g["trailing_mean"].notna()]
        te = g[(g["Date"] >= q0) & (g["Date"] <= q1)]
        label = f"{'chain-pooled' if pooled else 'per store'} | {qname}"
        for name, k in [("rule @ 90% fill (fit on train)", fit_multiplier(tr, target_fill=0.90)),
                        ("rule, profit-maximising k", fit_multiplier(tr))]:
            r = simulate_plan(te.demand, trailing_mean_rule(te.trailing_mean, k), te.unit_margin, te.unit_cost)
            rows.append({"grain | quarter": label, "plan": f"{name} (k={k:.2f})", **r.as_dict()})
        for ret in RETENTION:
            r = simulate_plan(te.demand, 0, te.unit_margin, te.unit_cost, preorder_retention=ret)
            rows.append({"grain | quarter": label, "plan": f"make-to-order, {ret:.0%} of demand retained", **r.as_dict()})
        perfect = simulate_plan(te.demand, te.demand, te.unit_margin, te.unit_cost)
        rows.append({"grain | quarter": label, "plan": "perfect foresight (ceiling)", **perfect.as_dict()})

sizing = pd.DataFrame(rows).set_index(["grain | quarter", "plan"])
display(sizing[["demand", "fill_rate", "waste_share_of_baked", "net_contribution"]]
        .style.format({"demand": "{:,.0f}", "fill_rate": "{:.0%}", "waste_share_of_baked": "{:.0%}",
                       "net_contribution": "₹{:,.0f}"}, na_rep="–"))

In [6]:
# How intermittent is demand at the level a baker actually decides (product x day)?
from bakery.features import demand_grid
for pooled in (False, True):
    g = demand_grid(fresh, item="Product", pooled=pooled)
    print(f"{'chain-pooled' if pooled else 'per store':12s}: {len(g):,} product-days, "
          f"{(g.demand > 0).mean():.1%} with any demand, mean {g.demand.mean():.2f} units/day "
          f"({g.loc[g.demand > 0, 'demand'].mean():.2f} when > 0)")

per store   : 48,246 product-days, 21.6% with any demand, mean 0.53 units/day (2.44 when > 0)
chain-pooled: 16,082 product-days, 49.5% with any demand, mean 1.58 units/day (3.19 when > 0)


**What the sizing shows:**
1. **Baking ahead with a simple rule loses heavily.** Hitting about 90% fill with a trailing-mean rule throws away 72–89% of what is baked, losing ₹0.25M–₹1.06M per quarter.
2. **The most profitable simple rule is to bake nothing** (k = 0), both per store and pooled. A single store sells a given fresh product on only about 1 day in 5, so a unit baked "just in case" is wasted more often than sold.
3. **Make-to-order / pre-order** earns its retention share of the ceiling: 50–90% of the perfect-foresight contribution (about ₹45k–₹84k per quarter), depending on how many customers are willing to order ahead. That share is unknown and will vary a lot by product. People pre-order birthday cakes, but rarely a donut.
4. The **perfect-foresight ceiling** is about ₹90k per quarter. The gap between "bake nothing" (₹0) and the ceiling is the value at stake.

#### Business objectives (revised)
- **BO1, fulfilment mode:** for each of the 22 fresh products, decide whether to **bake ahead**, **make to order / pre-order**, or run a **hybrid** (small display stock + pre-orders).
- **BO2, quantities:** for bake-ahead and hybrid products, set daily quantities per store (or centrally) that maximise net contribution.
- **BO3, profit:** raise fresh **net contribution** (margin on units sold − cost of discarded units) above the best simple alternative.
- **BO4, service:** keep peak days (weekends, festivals) from turning into frequent sell-outs.
- **BO5, demand shaping:** use cross-selling and customer segments to support pre-ordering and bundles of fresh items.

Out of scope: pricing, staffing, long-life stock replenishment, store openings and closures.

#### Business success criteria
Measured on **two out-of-sample festive quarters** (Q4 2023 and Q4 2024; rolling origin). Every plan is scored with the same simulator (`bakery.policy.simulate_plan`, demand = `Quantity`). The **reference plan** for each product is the best simple alternative among bake nothing, the profit-maximising trailing-mean rule, and make-to-order, all under the same pre-order assumptions.

| ID | Criterion | Target |
|---|---|---|
| BSC1 | Net contribution of the recommended plan vs the reference plan | Higher by ≥ 5 percentage points of the perfect-foresight ceiling (≈ ₹4.5k/quarter) in **both** quarters, with a 95% block-bootstrap CI (over days) for the difference that excludes 0 |
| BSC2 | Waste on bake-ahead products at a matched fill rate | ≥ 20% fewer wasted units than the trailing-mean rule at the same fill rate |
| BSC3 | Peak-day service | Fill rate on weekends and festival days no more than 5 pp below other days. Contribution-vs-fill trade-off reported at 50/70/90% fill so management can choose |
| BSC4 | Usability | A store manager sees tomorrow's mode and quantity for every fresh product on one screen, with the reason (demo in the app) |
| BSC5 | Demand-shaping insight | ≥ 3 recommendations (bundles, pre-order targeting), each backed by statistics that hold in both 2023 and 2024 |

### 1.2 Assess situation

**CRISP-DM outputs:** Inventory of resources · Requirements, assumptions & constraints · Risks & contingencies · Terminology · Costs & benefits

#### Inventory of resources
| Type | Resource |
|---|---|
| Data | One extract: 19,321 order lines × 40 columns, 2023–2024, 3 stores (`data/README.md`). Kaggle source not yet confirmed (see R7). No production log, inventory system, pre-order history or weather feed |
| People | One analyst (the author). **No access to the bakery**, so the business parameters (A0, A7, cost of waste) are the analyst's explicit assumptions and are sensitivity-tested. Reviewers: course instructor + an independent critic |
| Compute / software | Laptop + Google Colab; Python 3.12, pandas, scikit-learn, LightGBM, statsmodels, mlxtend, datasketch, Streamlit (all open source) |

#### Requirements
- **R-1, information available the evening before:** calendar, the public festival calendar and sales history up to the previous day. Not allowed as model inputs: `Promotion_Applied`, which varies within a store-day on 40% of store-days and so is a per-transaction attribute rather than a planned promotion; and on-the-day fields (`Units_Sold`, `Unsold_Units`, `Stock_Available`, `Waste_Pct`).
- **R-2, weather:** `Weather`/`Temperature` are the *actual* weather, identical across stores for each date. The primary model excludes them. A with-weather model is reported only as an upper bound.
- **R-3, reproducibility:** fixed seeds, versioned code, a notebook that runs top to bottom in Colab with outputs saved.
- **R-4, privacy:** customer data used only in aggregate (segments). No individual targeting outside the analysis.

#### Assumptions
| # | Assumption | Testable? | How it's handled |
|---|---|---|---|
| A0 | Current practice is to bake ahead every fresh product daily per store, using recent sales | No (no production log) | Proxied by the trailing-mean rule. Recorded fields are not used |
| A1 | Demand = `Quantity` (units ordered and paid for) | Partly. Bills and profit are computed from it on 100% of rows, but `Units_Sold > Quantity` on 27% of rows | **Decision gate in 2.4:** if `Units_Sold` turns out to be meaningful demand, §1 figures are recomputed |
| A2 | Recorded sales are uncensored demand (no hidden lost sales) | **No.** Transaction data can't show customers who found the shelf empty | Stated as a limitation. Under the simulator, unmet demand is explicit |
| A3 | Unsold fresh units are discarded (salvage = 0) | No | `config.SALVAGE_VALUE`, sensitivity-tested |
| A5 | 2023–2024 represents the near future; festivals recur | Partly | Rolling-origin evaluation over two quarters |
| A6 | Unsold long-life units are sold later, so they aren't waste | No | Long-life products are out of scope |
| A7 | **Pre-order retention:** the share of a product's demand kept if it's only made to order | No (no pre-order history) | Scenarios: cakes 50/70/90%, other fresh 0/20/40%. Recommendations must hold across them |

#### Constraints
A single static extract (no new data collection). No stakeholder interviews. One analyst, an academic timeline (see 1.4) and free compute.

#### Risks & contingencies
| # | Risk | Likelihood | Impact | Contingency |
|---|---|---|---|---|
| R1 | Recorded production/unsold fields are generated per order line, not real production | **Confirmed** | High | Not used. The bake decision is simulated from `Quantity` only (§1.1) |
| R2 | The dataset looks **synthetic** (shelf life varies within a product; ratings flat across segments) | High | Medium | Benchmark every pattern against naive baselines. Report unsupported patterns. Frame results as a method demonstration |
| R3 | **Very intermittent demand:** about 22% of store × product × days have any sales | Confirmed | High | Intermittent-demand methods (Croston/TSB, zero-inflated / Poisson models). Pooled vs per-store comparison. Category-level forecast + product disaggregation as a fallback |
| R4 | **Leakage** from fields recorded on the day | Medium | High | R-1/R-2 exclusions. Leakage audit in Phase 3 |
| R5 | Few festival examples (each festival appears twice) | Confirmed | Medium | Festival flags and days-to-festival features rather than date effects. Evaluate on both Q4s |
| R6 | One test quarter could be luck | Medium | Medium | Two rolling-origin quarters + block-bootstrap CIs (BSC1) |
| R7 | Dataset provenance not confirmed (source/licence) | Open | Low | Confirm the Kaggle URL with the project owner before publishing |
| R8 | The A7 retention assumptions drive the mode decision | High | High | Only recommend modes that hold across all A7 scenarios. Otherwise flag "needs a pre-order pilot" |
| R9 | The validation window (Jul–Sep 2024) has no Q4 festivals | Confirmed | Medium | Tune on rolling origins that include Q4 2023 |

#### Terminology
**Fresh product:** median shelf life ≤ 7 days · **Long-life product:** median shelf life ≥ 30 days · **Demand:** `Quantity` · **Bake-ahead / make-to-order / hybrid:** stock baked the evening/morning before / produced only against a pre-order / small stock plus pre-orders for the rest · **Retention (A7):** share of demand kept under make-to-order · **Waste:** units baked ahead and not sold that day · **Fill rate:** units sold (from stock + pre-order) ÷ demand · **Net contribution:** margin on units sold − cost of discarded units · **Perfect-foresight ceiling:** net contribution if exactly the demand were baked · **Critical ratio:** Cu / (Cu + Co) = margin ÷ price · **Pinball loss:** loss function for quantile forecasts · **WAPE:** Σ|actual − forecast| ÷ Σ actual · **Croston/TSB:** forecasting methods for intermittent demand · **Support / confidence / lift:** share of orders containing an itemset / P(B|A) / confidence ÷ P(B) · **RFM:** recency, frequency, monetary value.

#### Costs & benefits
Unit economics (fit on training data only, Jan 2023 – Jun 2024): **Co** (cost of waste) = unit cost; **Cu** (cost of a stockout) = unit margin. Critical ratio = Cu / (Cu + Co) = margin ÷ price.

In [7]:
# Unit economics and critical ratios, training period only (no test-quarter information)
train = fresh[fresh["Date"] <= config.TRAIN_END]
econ = (train.assign(margin_x_q=(train.Selling_Price - train.Cost_Price) * train.Quantity,
                     cost_x_q=train.Cost_Price * train.Quantity)
              .groupby("Category")[["margin_x_q", "cost_x_q", "Quantity"]].sum())
econ["avg_unit_margin"] = econ.margin_x_q / econ.Quantity
econ["avg_unit_cost"] = econ.cost_x_q / econ.Quantity
econ["critical_ratio"] = econ.avg_unit_margin / (econ.avg_unit_margin + econ.avg_unit_cost)
display(econ[["avg_unit_margin", "avg_unit_cost", "critical_ratio"]].sort_values("critical_ratio").round(2))
print("A unit baked ahead pays off only if P(demand covers it) > 1 - critical ratio "
      f"(≈ {1 - econ.critical_ratio.max():.2f}–{1 - econ.critical_ratio.min():.2f} here).")

,avg_unit_margin,avg_unit_cost,critical_ratio
Category,,,
Dairy & Spreads,10.22,28.17,0.27
Cakes,124.15,202.50,0.38
Bread & Daily,10.10,12.65,0.44
Beverages,28.36,34.07,0.45
Savoury Bakery,11.36,12.48,0.48
Pastry & Sweet,19.94,21.39,0.48


A unit baked ahead pays off only if P(demand covers it) > 1 - critical ratio (≈ 0.52–0.73 here).


**Benefits (conditional; sized from the table above):**
- The value at stake is the gap between the best simple plan and the perfect-foresight ceiling, about ₹90k per quarter (roughly ₹0.35M a year in the festive-quarter run rate).
- Compared with the assumed current practice (A0, bake ahead at about 90% fill), simply **choosing the right mode per product** is worth hundreds of thousands of rupees per quarter.
- The **data mining contribution** is the part above the best simple plan. That is what BSC1 measures. These are conditional estimates, not forecasts: they depend on A0 and A7.

**Costs:** about 60–80 analyst-hours for this project (≈ ₹60–80k at an assumed ₹1,000/hour), open-source software, and free hosting (Streamlit Community Cloud). Running costs: a monthly retraining check (Phase 6). A pre-order pilot to measure A7 is recommended before full roll-out (Phase 5/6).

### 1.3 Determine data mining goals

**CRISP-DM outputs:** Data mining goals · Data mining success criteria

| ID | Data mining goal | Technique | Data mining success criterion | Supports |
|---|---|---|---|---|
| DM1 | **Forecast the next-day demand distribution** per fresh product × store (and pooled), as quantiles 0.5 / 0.6 / 0.7 / 0.8 / 0.9 and P(demand ≥ 1) | Intermittent-demand baselines (seasonal-naive, trailing mean, Croston/TSB) vs count / quantile models (Poisson / zero-inflated GLM, LightGBM Tweedie & quantile) | Mean pinball loss ≥ 10% lower than the best baseline on both test quarters. Empirical coverage of the 0.7/0.8/0.9 quantiles within ±5 pp. Brier score for P(demand ≥ 1) better than baseline | BO1–BO4 |
| DM2 | **Choose the mode and bake quantity** per product × day: bake at the critical-ratio quantile, with make-to-order for the rest; mode set per product from validation contribution across the A7 scenarios | Newsvendor policy + scenario analysis, tuned on rolling origins | Meets BSC1–BSC3 | BO1–BO4 |
| DM3 | **Bundles:** products bought together | FP-Growth association rules | ≥ 5 rules with support ≥ 1% of orders and lift ≥ 1.5 in **both** 2023 and 2024. ≥ 2 rules where a fresh product is the consequent | BO5, BSC5 |
| DM4 | **Customer segments for pre-order targeting** | k-means / hierarchical clustering on RFM + fresh/cake share + festival buying | 3–6 segments, silhouette ≥ 0.25, ARI ≥ 0.8 across 10 seeds, each segment ≥ 5% of customers, and ≥ 1 segment whose cake share is ≥ 1.5× the average | BO5, BSC5 |
| DM5 (appendix) | **Scalable "customers like you" lookup** (course LSH topic) | MinHash LSH on customer baskets | Recall@10 ≥ 0.9 vs exact Jaccard. Included as a scalability demo: with 500 customers exact search is trivial, so this has no business success criterion | Course requirement |

**Outlier analysis** (course template) is a supporting step in Phase 3: unusually large orders and festival spikes are checked and handled before modeling.

### 1.4 Produce project plan

**CRISP-DM outputs:** Project plan · Initial assessment of tools & techniques

Dates assume work starts 2026-11-03. **The assignment due date still needs confirming**, and the dates will be adjusted to fit it.

| Phase | Planned dates | Effort | Inputs → outputs | Milestone / go-no-go |
|---|---|---|---|---|
| 1 Business Understanding | Nov 3–4 | 10% | Raw data, course material → §1 | Critic: no major issues |
| 2 Data Understanding | Nov 4–6 | 20% | §1 → data reports; **A1 decision gate** | If A1 fails, revise §1 figures |
| 3 Data Preparation | Nov 6–9 | 25% | §2 → demand grids, features, RFM, baskets; leakage audit | Leakage audit passes |
| 4 Modeling | Nov 9–12 | 20% | §3 → DM1–DM5 models | DM criteria met, or loop back to 3 |
| 5 Evaluation | Nov 12–13 | 10% | §4 → BSC1–BSC5 backtest, go/no-go | Go/no-go decision |
| 6 Deployment | Nov 13–16 | 15% | §5 → app, monitoring, report, article, video script | Final critic review |

**Resources per phase:** one analyst for all phases, with a critic review at the end of each (≥ 2 rounds).

**Initial assessment of tools & techniques:**
- **pandas / scikit-learn / statsmodels:** baselines, GLMs (Poisson, zero-inflated), clustering, metrics.
- **Croston/TSB:** short custom implementations, because demand is mostly zeros.
- **LightGBM:** Tweedie and quantile objectives for sparse counts.
- **mlxtend:** FP-Growth.
- **datasketch:** MinHash LSH.
- **Streamlit:** a quick way to put a usable tool in front of staff.
- **IBM SPSS Modeler** (suggested in the assignment) has a built-in CRISP-DM project view, but it needs a licence and can't be shared as a runnable notebook, so the CRISP-DM structure is followed in this notebook and report instead.

> **Critic review: Phase 1.** The revisions are logged in `critiques/phase1_business_understanding_round*.md`.
> The phase is finished when every output above is filled in and the latest critic round lists no major issues.

---
# Phase 2 · Data Understanding

### 2.1 Collect initial data

**CRISP-DM outputs:** Initial data collection report

In [8]:
raw = load_raw()
print(raw.shape)
raw.head()

(19321, 40)


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,Customer_Segment,Loyalty_Member,Date,Year,Month,Day,Day_of_Week,Weekend,Hour,Time,Season,Festival,Weather,Temperature,Product,Category,Quantity,Unit_Price,Cost_Price,Discount_Percentage,Selling_Price,Total_Bill,Payment_Method,Promotion_Applied,Promotion_Type,Shelf_Life_Days,Storage_Type,Perishability,Store_ID,Units_Produced,Units_Sold,Unsold_Units,Stock_Available,Waste_Pct,Profit,Customer_Rating
0,ORD00001,CUST0328,46,Other,Young Professional,True,2023-01-01,2023,1,1,Sunday,True,8,8:47,Winter,New Year,Foggy,21.2,Pastry,Pastry & Sweet,6,62.49,29.72,14.7,53.30,319.80,Cash,True,Festival,3,Refrigerated,Very High,ST02,8,6,2,2,25.0,141.48,4
1,ORD00001,CUST0328,46,Other,Young Professional,True,2023-01-01,2023,1,1,Sunday,True,8,8:47,Winter,New Year,Foggy,21.2,Biscuit,Indian Bakery,3,30.14,12.63,13.9,25.95,77.85,Cash,True,Festival,64,Dry Storage,Low,ST02,4,3,1,1,25.0,39.96,4
2,ORD00002,CUST0016,21,Female,Student,False,2023-01-01,2023,1,1,Sunday,True,15,15:14,Winter,New Year,Foggy,21.2,Chocolate Cake,Cakes,1,425.49,256.56,15.3,360.39,360.39,UPI,True,Festival,3,Refrigerated,Very High,ST03,3,2,1,1,33.3,103.83,4
3,ORD00002,CUST0016,21,Female,Student,False,2023-01-01,2023,1,1,Sunday,True,15,15:14,Winter,New Year,Foggy,21.2,Nankhatai,Indian Bakery,7,30.29,13.18,18.9,24.57,171.99,UPI,True,Festival,23,Dry Storage,Low,ST03,13,10,3,2,23.1,79.73,4
4,ORD00002,CUST0016,21,Female,Student,False,2023-01-01,2023,1,1,Sunday,True,15,15:14,Winter,New Year,Foggy,21.2,Paav,Bread & Daily,3,7.94,3.64,11.9,7.00,21.00,UPI,True,Festival,1,Room Temperature,Very High,ST03,4,3,1,1,25.0,10.08,4


In [9]:
# TODO: Collection report: source, licence, checksum, grain, coverage, how it was obtained, any problems.

### 2.2 Describe data

**CRISP-DM outputs:** Data description report

In [10]:
# TODO: Types, ranges, cardinalities and units for every column. Explain the grain: order line vs order vs store-day.

### 2.3 Explore data

**CRISP-DM outputs:** Data exploration report

In [11]:
# TODO: EDA: demand over time, weekday/festival/season/weather effects, category & store mix, price/discount/promo, basket size and co-occurrence, customer behaviour, waste patterns.

### 2.4 Verify data quality

**CRISP-DM outputs:** Data quality report

In [12]:
# TODO: Missing values, duplicates, arithmetic checks (sold+unsold=produced, price x qty = bill, discount math), date/calendar consistency, signs of synthetic data.

> **Critic review: Phase 2.** The revisions are logged in `critiques/phase2_data_understanding_round*.md`.
> The phase is finished when every output above is filled in and the latest critic round lists no major issues.

---
# Phase 3 · Data Preparation

### 3.1 Select data

**CRISP-DM outputs:** Rationale for inclusion / exclusion

In [13]:
# TODO: Choose rows and columns for each modeling goal, justify each exclusion, and flag leakage columns.

### 3.2 Clean data

**CRISP-DM outputs:** Data cleaning report

In [14]:
# TODO: Handle the issues found in 2.4. Outlier analysis on orders (IQR + Isolation Forest) and how each outlier is handled.

### 3.3 Construct data

**CRISP-DM outputs:** Derived attributes · Generated records

In [15]:
# TODO: Store x category x day demand table (zero-fill days with no sales), calendar/festival/weather features, lag and rolling features, RFM table, basket matrix.

### 3.4 Integrate data

**CRISP-DM outputs:** Merged data

In [16]:
# TODO: Join the demand table with calendar and weather attributes. Join the customer RFM table with demographics.

### 3.5 Format data

**CRISP-DM outputs:** Reformatted data · Dataset · Dataset description

In [17]:
# TODO: sklearn Pipeline/ColumnTransformer. Save processed datasets to data/processed/ and document them.

> **Critic review: Phase 3.** The revisions are logged in `critiques/phase3_data_preparation_round*.md`.
> The phase is finished when every output above is filled in and the latest critic round lists no major issues.

---
# Phase 4 · Modeling

### 4.1 Select modeling technique

**CRISP-DM outputs:** Modeling technique · Modeling assumptions

In [18]:
# TODO: Demand: seasonal-naive baseline, linear/Poisson, random forest, LightGBM (incl. quantile). Basket: Apriori/FP-Growth. Segments: k-means/hierarchical. Optional: MinHash LSH.

### 4.2 Generate test design

**CRISP-DM outputs:** Test design

In [19]:
# TODO: Time-based train/valid/test split (config.TRAIN_END / VALID_END), rolling-origin CV, metrics (MAE, WAPE, pinball loss), plus a leakage check.

### 4.3 Build model

**CRISP-DM outputs:** Parameter settings · Models · Model descriptions

In [20]:
# TODO: Fit and tune each technique. Turn forecasts into bake quantities with policy.critical_ratio. Mine rules and fit segments.

### 4.4 Assess model

**CRISP-DM outputs:** Model assessment · Revised parameter settings

In [21]:
# TODO: Comparison table against the baseline, error analysis by store/category/festival, rule quality (support/confidence/lift), segment quality (silhouette, profiles).

> **Critic review: Phase 4.** The revisions are logged in `critiques/phase4_modeling_round*.md`.
> The phase is finished when every output above is filled in and the latest critic round lists no major issues.

---
# Phase 5 · Evaluation

### 5.1 Evaluate results

**CRISP-DM outputs:** Assessment of results w.r.t. business success criteria · Approved models

In [22]:
# TODO: Backtest the bake policy against historical production on the test period: waste, fill rate, profit. Check against the targets in config.

### 5.2 Review process

**CRISP-DM outputs:** Review of process

In [23]:
# TODO: Audit every phase for missed steps, leakage and questionable decisions.

### 5.3 Determine next steps

**CRISP-DM outputs:** List of possible actions · Decision

In [24]:
# TODO: Go/no-go for deployment, with reasons.

> **Critic review: Phase 5.** The revisions are logged in `critiques/phase5_evaluation_round*.md`.
> The phase is finished when every output above is filled in and the latest critic round lists no major issues.

---
# Phase 6 · Deployment

### 6.1 Plan deployment

**CRISP-DM outputs:** Deployment plan

In [25]:
# TODO: Save the final pipeline to models/. Describe the Streamlit Daily Production Planner (deployment/app.py) and how staff would use it.

### 6.2 Plan monitoring & maintenance

**CRISP-DM outputs:** Monitoring & maintenance plan

In [26]:
# TODO: Drift and forecast-error alerts, retraining trigger and schedule (see deployment/monitoring_plan.md).

### 6.3 Produce final report

**CRISP-DM outputs:** Final report · Final presentation

In [27]:
# TODO: Write up reports/final_report.md, the Medium article and the video script.

### 6.4 Review project

**CRISP-DM outputs:** Experience documentation

In [28]:
# TODO: Lessons learned: what worked, what didn't, what we'd do differently.

> **Critic review: Phase 6.** The revisions are logged in `critiques/phase6_deployment_round*.md`.
> The phase is finished when every output above is filled in and the latest critic round lists no major issues.